#Interactive pairwise significant calculator.

Function definition: Uses Fisher's Exact test or Two-proportion Z-test where appropriate.

In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd

# Global definition of dataset group sizes
DATASET_SIZES = {
    'HPC': 15,
    'None': 40,
    'Strictly Edge': 77,
    'Edge': 106,
    'FaaS': 224,
    'Serverless': 352
}


def calculate_pairwise_thresholds(baseline_prevalence=0.30):
    """
    Calculates the exact percentage point differences required for significance
    across all 15 pairwise comparisons using a given baseline prevalence.
    """
    results = []
    group_names = list(DATASET_SIZES.keys())

    for i in range(len(group_names)):
        for j in range(i + 1, len(group_names)):
            g1, g2 = group_names[i], group_names[j]
            n1, n2 = DATASET_SIZES[g1], DATASET_SIZES[g2]

            # Determine the realistic discrete count for Group A at the baseline
            x1 = round(baseline_prevalence * n1)
            p1_actual = x1 / n1

            # 1. Fisher's Exact Test Boundaries (for comparisons involving small groups)
            fisher_low, fisher_high = None, None
            for x2 in range(n2 + 1):
                _, p_val = stats.fisher_exact([[x1, n1 - x1], [x2, n2 - x2]])
                if p_val < 0.05:
                    if x2 / n2 < p1_actual:
                        fisher_low = x2
                    elif x2 / n2 > p1_actual and fisher_high is None:
                        fisher_high = x2

            f_diff_low = (p1_actual - (fisher_low / n2)) * 100 if fisher_low is not None else None
            f_diff_high = ((fisher_high / n2) - p1_actual) * 100 if fisher_high is not None else None

            # 2. Two-Proportion Z-Test Boundaries (for larger groups)
            z_low, z_high = None, None
            for x2 in range(n2 + 1):
                p2 = x2 / n2
                if p2 == p1_actual:
                    continue
                p_pool = (x1 + x2) / (n1 + n2)
                if p_pool == 0 or p_pool == 1:
                    continue

                z_stat = (p1_actual - p2) / np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
                p_val_z = 2 * (1 - stats.norm.cdf(abs(z_stat)))

                if p_val_z < 0.05:
                    if p2 < p1_actual:
                        z_low = x2
                    elif p2 > p1_actual and z_high is None:
                        z_high = x2

            z_diff_low = (p1_actual - (z_low / n2)) * 100 if z_low is not None else None
            z_diff_high = ((z_high / n2) - p1_actual) * 100 if z_high is not None else None

            # Select recommended test rule: Fisher if either group is small (< 50)
            use_fisher = (n1 < 50 or n2 < 50)
            final_low = f_diff_low if use_fisher else z_diff_low
            final_high = f_diff_high if use_fisher else z_diff_high

            range_str = f"-{final_low:.1f}% / +{final_high:.1f}%" if (final_low and final_high) else f"+{final_high:.1f}%"
            conservative_max = max(final_low or 0, final_high or 0)

            results.append({
                'Group A': f"{g1} ({n1})",
                'Group B': f"{g2} ({n2})",
                'Test Used': "Fisher's Exact" if use_fisher else "Two-Proportion Z",
                'Required Difference Range': range_str,
                'Conservative Threshold': f"{conservative_max:.1f}%"
            })

    return pd.DataFrame(results)


def calculate_group_confidence_intervals(observed_counts, alpha=0.05):
    """
    Calculates Clopper-Pearson exact 95% confidence intervals for all groups
    based on a passed dictionary of observed counts.
    """
    results = []

    for name, N in DATASET_SIZES.items():
        n = observed_counts.get(name, 0)
        prevalence = (n / N) * 100

        # Clopper-Pearson exact calculation using Beta distribution bounds
        lower_bound = stats.beta.ppf(alpha / 2, n, N - n + 1) if n > 0 else 0
        upper_bound = stats.beta.ppf(1 - alpha / 2, n + 1, N - n) if n < N else 1

        lower_bound_pct = lower_bound * 100
        upper_bound_pct = upper_bound * 100

        results.append({
            'Group': name,
            'N': N,
            'Observed (n)': n,
            'Prevalence (%)': f"{prevalence:.1f}%",
            '95% Confidence Interval (Range approach)': f"[{lower_bound_pct:.1f}% to {upper_bound_pct:.1f}%]"
        })

    return pd.DataFrame(results)


# Run the comparison function, *interactively*

**IMPORTANT:** Below, you must Change the baseline prevalence in the code and run it. The two variables to set are _prevalence1_ (for Group A) and _prevalence2_ (for Group B), with the observed prevalence of service X, in the two groups you want to compare.

How to interpret results? In the table, search for the line with the two Groups you want to compare (the ones for which you configured the empirical prevalence of a service).

This code was used to generate during our analysis, to ensure only statistically significant differences are reported when comparing prevalence of a service (or type of service, category, etc. between two groups).

In [ ]:
# ==========================================
# Call the functions with the paper data
# Pairwise significance thresholds
# ==========================================
print("--- FUNCTION 1: PAIRWISE SIGNFICANCE THRESHOLDS ---")
prevalence1 = 0.267
prevalence2 = 0.481
for baseline_prevalence in [prevalence1]:
    print("\nBaseline prevalente = %f" % (baseline_prevalence))
    pairwise_df = calculate_pairwise_thresholds(baseline_prevalence)
    print(pairwise_df.to_string(index=False))

print("\n" + "="*80 + "\n")

difference = (prevalence1 - prevalence2) * 100
print("\n Difference%.2f" % difference)


--- FUNCTION 1: PAIRWISE SIGNFICANCE THRESHOLDS ---

Baseline prevalente = 0.267000
           Group A            Group B        Test Used Required Difference Range Conservative Threshold
          HPC (15)          None (40)   Fisher's Exact           -21.7% / +33.3%                  33.3%
          HPC (15) Strictly Edge (77)   Fisher's Exact           -20.2% / +29.2%                  29.2%
          HPC (15)         Edge (106)   Fisher's Exact           -19.1% / +30.9%                  30.9%
          HPC (15)         FaaS (224)   Fisher's Exact           -18.2% / +28.7%                  28.7%
          HPC (15)   Serverless (352)   Fisher's Exact           -17.6% / +28.2%                  28.2%
         None (40) Strictly Edge (77)   Fisher's Exact           -15.8% / +19.3%                  19.3%
         None (40)         Edge (106)   Fisher's Exact           -15.2% / +19.7%                  19.7%
         None (40)         FaaS (224)   Fisher's Exact           -13.7% / +18.0%    

#Single service Clopper-Pearson Confidence Intervals
Function definition and configurations for comparisons used during our analysis in the paper.

In [ ]:
# ==========================================
# Call the functions with the paper data
# Single-service Clopper-Pearson confidence
# intervals
# ==========================================

print("--- FUNCTION 2: SINGLE SERVICE CLOPPER-PEARSON CONFIDENCE INTERVALS ---")
# S3 (Fig. 4)
previous_counts = {
    'HPC': 11,
    'Edge': 81,
    'Serverless': 253,
    #'None': 0,
    'Strictly Edge': 56,
    'FaaS': 153
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print("\nS3")
print(ci_df.to_string(index=False))

# EC2 (Fig. 4)
print("\nEC2")
previous_counts = {
    'HPC': 11,
    'Edge': 37,
    'Serverless': 132,
    #'None': 0,
    'Strictly Edge': 27,
    'FaaS': 69
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

# Lambda (Fig. 4)
print("\nLambda")
previous_counts = {
    'HPC': 8,
    'Edge': 64,
    'Serverless': 220,
    #'None': 0,
    'Strictly Edge': 50,
    'FaaS': 220
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

# Others for HPC
print("\nFSX")
previous_counts = {
    'HPC': 6
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nEKS")
previous_counts = {
    'HPC': 5
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nBatch / DynamoDB")
previous_counts = {
    'HPC': 4,
    'Serverless': 123
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

# Others for Edge
print("\nCloudFront")
previous_counts = {
    'HPC': 4,
    'Edge': 45,
    'Serverless': 43,
    'Strictly Edge': 16
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nThirdParty")
previous_counts = {
    'HPC': 8,
    'Edge': 34
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nUserConsumerWeb")
previous_counts = {
    'Edge': 28
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nUserConsumerMobile")
previous_counts = {
    'Edge': 26
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nUserConsumerEdge")
previous_counts = {
    'Edge': 21
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

# Others for Serverless
print("\nAPIGateway")
previous_counts = {
    'Edge': 34,
    'Serverless': 96,
    'Strictly Edge': 25,
    'FaaS': 80
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nRDS")
previous_counts = {
    'Serverless': 71
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

# Storage (Fig. 7)
print("\nObject")
previous_counts = {
    'HPC': 11,
    'Edge': 81,
    'Serverless': 253,
    'None': 0,
    'Strictly Edge': 56,
    'FaaS': 153
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nFile")
previous_counts = {
    'HPC': 7,
    'Edge': 2,
    'Serverless': 1,
    'None': 0,
    'Strictly Edge': 1,
    'FaaS': 6
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nNoSQL")
previous_counts = {
    'HPC': 5,
    'Edge': 52,
    'Serverless': 149,
    'None': 3,
    'Strictly Edge': 35,
    'FaaS': 110
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nSQL")
previous_counts = {
    'HPC': 5,
    'Edge': 31,
    'Serverless': 107,
    'None': 12,
    'Strictly Edge': 18,
    'FaaS': 59
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

DATASET_SIZES = {
    'HPC': 15,
    'None': 40,
    'Strictly Edge': 77,
    'Edge': 106,
    'FaaS': 224,
    'Serverless': 352
}

# Goals (Table 4)
print("\nCompute intensive")
previous_counts = {
    'HPC': 8,
    'Edge': 18,
    'Serverless': 55,
    'None': 2,
    'Strictly Edge': 15,
    'FaaS': 36
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))

print("\nInteractive")
previous_counts = {
    'HPC': 5,
    'Edge': 47,
    'Serverless': 115,
    'None': 11,
    'Strictly Edge': 31,
    'FaaS': 84
}
ci_df = calculate_group_confidence_intervals(observed_counts=previous_counts)
print(ci_df.to_string(index=False))


--- FUNCTION 2: SINGLE SERVICE CLOPPER-PEARSON CONFIDENCE INTERVALS ---

S3
        Group   N  Observed (n) Prevalence (%) 95% Confidence Interval (Range approach)
          HPC  15            11          73.3%                         [44.9% to 92.2%]
         None  40             0           0.0%                           [0.0% to 8.8%]
Strictly Edge  77            56          72.7%                         [61.4% to 82.3%]
         Edge 106            81          76.4%                         [67.2% to 84.1%]
         FaaS 224           153          68.3%                         [61.8% to 74.3%]
   Serverless 352           253          71.9%                         [66.9% to 76.5%]

EC2
        Group   N  Observed (n) Prevalence (%) 95% Confidence Interval (Range approach)
          HPC  15            11          73.3%                         [44.9% to 92.2%]
         None  40             0           0.0%                           [0.0% to 8.8%]
Strictly Edge  77            27        